<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/ejercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Ejercicio 1.5 · Más variables y la temporada

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Antes de empezar, guarda tu copia: **Archivo → Guardar una copia en Drive**.

Escribe tu respuesta en cada celda de código y ejecútala con **Shift + Enter**.

### Prepara los datos

Ejecuta estas cuatro celdas. Ya dejan listas `entreno`, `prueba`, el `indice` de cada mes y la función `preparar_x`.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [ ]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [ ]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [ ]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

entreno = tabla[tabla["mes"] < "2025-07"]
prueba = tabla[tabla["mes"] >= "2025-07"].copy()

print(len(entreno), len(prueba))

ventas_2024 = ventas[ventas["fecha"].str[:4] == "2024"]
por_mes = ventas_2024.groupby(ventas_2024["fecha"].str[5:7].astype(int))["total"].sum()

indice = por_mes / por_mes.mean()

indice.round(2)

### 1. La regla con índice, por mes

Crea en `prueba` la columna `regla_indice`, como en la clase. El MAE debe dar `827`. Luego calcula su error por `numero_mes`, igual que en el 1.4. Diciembre debe dar cerca de `1028`. En el 1.4, sin índice, daba `1031`: cliente por cliente, casi no cambia. Veremos por qué en el ejercicio 4.

In [ ]:
# Escribe tu código aquí

### 2. El modelo

Entrena la regresión con `mes_anterior` y `promedio_3` como `numeros`, y `tipo`, `ciudad` y `numero_mes` como `textos`. Guarda su predicción en `prueba["regresion"]`. El MAE debe dar `921`.

In [ ]:
# Escribe tu código aquí

### 3. Una variable más

Crea otra lista, `numeros_6 = numeros + ["promedio_6"]`, y entrena otro modelo. No guardes su predicción en `prueba["regresion"]`, porque la usaremos después. Su MAE debe dar `893`. ¿Le gana a la regla con índice? Responde en un comentario.

In [ ]:
# Escribe tu código aquí

### 4. Por cartera

Suma `total`, `regla_indice` y `regresion` por `id_vendedor` y `mes`. Calcula el MAE de cada método. Deben dar `4963` y `4819`.

¿Por qué el error de cada cliente casi no mejora, y el de la cartera sí? Responde en un comentario.

> 💡 Para ver dónde está la mejora, calcula el error de cartera por mes, con `error.groupby("mes").mean()`.

In [ ]:
# Escribe tu código aquí

### 5. Los clientes sin compras

La regresión les predice compras a clientes que ya no compran. Crea una copia de la predicción que les ponga cero cuando `promedio_3` es 0:

```python
prueba["regresion_cero"] = prueba["regresion"].where(prueba["promedio_3"] > 0, 0)
```

`.where()` deja el valor si se cumple la condición, y si no, usa el 0. El MAE debe bajar a `861`. ¿Es trampa usar `promedio_3` aquí? Responde en un comentario.

In [ ]:
# Escribe tu código aquí

### 6. ¿Hacen falta todas?

Entrena el modelo sin `tipo` ni `ciudad`: solo `numero_mes` en `textos`. Debe dar `916` por cliente. ¿Te quedarías con este modelo? Responde en un comentario.

In [ ]:
# Escribe tu código aquí

## 🎉 ¡Terminaste!

Compara tus respuestas con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0